# MODUL PRAKTIKUM 2
## DATA WAREHOUSE DAN SKEMA STAR & SNOWFLAKE DENGAN PYTHON
**Mata Kuliah:** Data Mining  
**Program Studi:** S1 Teknik Informatika, Universitas Semarang (USM)  
**Waktu Pelaksanaan:** 13 Oktober 2026

---

### Tujuan Praktikum
Setelah menyelesaikan praktikum ini, mahasiswa diharapkan mampu:
1. Memahami konsep Data Warehouse serta perbedaan tabel fakta (*fact table*) dan tabel dimensi (*dimension table*).
2. Membangun tabel fakta dan tabel dimensi menggunakan library **Pandas**.
3. Merancang dan mengimplementasikan **Skema Star** dengan melakukan penggabungan (*join/merge*) antar tabel.
4. Merancang dan mengimplementasikan **Skema Snowflake** dengan dimensi ternormalisasi.
5. Melakukan operasi analisis **OLAP** sederhana (*roll-up, drill-down, slice, dice*) pada data warehouse.
6. Menyimpan sekumpulan tabel data warehouse ke basis data **SQLite** sebagai fondasi ETL.


---
### Dasar Teori Singkat
**Data Warehouse** adalah sistem basis data yang dirancang khusus untuk mendukung pelaporan analisis (*OLAP*) dan pengambilan keputusan manajerial, bukan untuk transaksi harian (*OLTP*).

Model data disusun menjadi:
- **Tabel Fakta (Fact Table):** Menyimpan ukuran/kegiatan inti yang diukur (*measure* numerik) dan kunci asing (*Foreign Key*) menuju dimensi.
- **Tabel Dimensi (Dimension Table):** Menyimpan atribut penjelas deskriptif konteks (*siapa, apa, kapan, di mana*).

#### Perbandingan Skema:
- **Skema Star:** Satu tabel fakta di tengah, langsung terhubung ke tabel dimensi yang **tidak dinormalisasi** (*denormalized*). Struktur sederhana dan query sangat cepat.
- **Skema Snowflake:** Variasi dari Star, di mana tabel dimensi **dinormalisasi** menjadi beberapa tabel bertingkat (misal dimensi produk dipecah lagi menjadi tabel kategori). Struktur hemat ruang penyimpanan namun membutuhkan join berantai.


---
## LANGKAH 1: MEMBANGUN TABEL FAKTA DAN DIMENSI DENGAN PANDAS
Sebelum membangun skema, kita siapkan terlebih dahulu tabel fakta (`sales`) dan tiga tabel dimensi (`dim_pelanggan`, `dim_produk`, `dim_waktu`).


In [ ]:
import pandas as pd
import numpy as np

# 1.1 Membangun Tabel Fakta (Fact Table)
sales = pd.DataFrame({
    "id_transaksi": [101, 102, 103, 104, 105, 106, 107, 108],
    "id_pelanggan": [1,   2,   1,   3,   2,   1,   3,   2],
    "id_produk":    [1,   2,   3,   1,   2,   3,   1,   2],
    "id_waktu":     [1,   1,   1,   2,   2,   2,   3,   3],
    "jumlah":       [2,   1,   5,   1,   4,   3,   2,   2],
    "harga_satuan": [80000, 1500000, 25000, 80000, 1500000, 25000, 80000, 1500000],
})

# Measure (ukuran bisnis) dihitung: total_harga = jumlah * harga_satuan
sales["total_harga"] = sales["jumlah"] * sales["harga_satuan"]
sales[["id_transaksi", "id_pelanggan", "id_produk", "id_waktu", "jumlah", "harga_satuan", "total_harga"]]


### 1.2 Membangun Tabel Dimensi (Dimension Tables)
Tabel dimensi memberikan konteks deskriptif terhadap fakta penjualan.


In [ ]:
# 1. Dimensi Pelanggan (Who & Where)
dim_pelanggan = pd.DataFrame({
    "id_pelanggan": [1, 2, 3],
    "nama":         ["Andi", "Budi", "Citra"],
    "kota":         ["Semarang", "Semarang", "Solo"],
    "provinsi":     ["Jawa Tengah", "Jawa Tengah", "Jawa Tengah"],
    "segmen":       ["Retail", "Wholesale", "Retail"],
})

# 2. Dimensi Produk (What)
dim_produk = pd.DataFrame({
    "id_produk":    [1, 2, 3],
    "nama_produk":  ["Laptop", "Printer", "Tinta"],
    "kategori":     ["Elektronik", "Elektronik", "Perlengkapan"],
    "harga_satuan": [8000000, 1500000, 25000],
})

# 3. Dimensi Waktu (When)
dim_waktu = pd.DataFrame({
    "id_waktu": [1, 2, 3],
    "tanggal":  ["2026-01-05", "2026-01-12", "2026-01-19"],
    "tahun":    [2026, 2026, 2026],
    "bulan":    [1, 1, 1],
    "kuartal":  ["Q1", "Q1", "Q1"],
    "hari":     ["Min", "Min", "Min"],
})

print("Dimensi Pelanggan:")
display(dim_pelanggan)
print("\nDimensi Produk:")
display(dim_produk)
print("\nDimensi Waktu:")
display(dim_waktu)


### 1.3 Melihat Struktur Data Warehouse
Memeriksa ukuran (*shape*) dan tipe data (*dtypes*) tabel sebelum operasi merge.


In [ ]:
print("Sales:", sales.shape, "| Pelanggan:", dim_pelanggan.shape,
      "| Produk:", dim_produk.shape, "| Waktu:", dim_waktu.shape)
print("\nTipe Data Sales:")
print(sales.dtypes)


---
## LANGKAH 2: IMPLEMENTASI SKEMA STAR
Pada skema Star, tabel fakta di tengah digabungkan (*merge*) langsung dengan tabel dimensi melalui foreign key.


In [ ]:
# Gabungkan fakta -> dimensi pelanggan -> dimensi waktu
star = sales.merge(dim_pelanggan, on="id_pelanggan")
star = star.merge(dim_waktu, on="id_waktu")

# Tampilkan kolom kunci hasil skema star
star[["id_transaksi", "nama", "kota", "segmen", "tahun", "bulan", "total_harga"]]


### Agregasi pada Skema Star


In [ ]:
# Total penjualan per kota
print("Total Penjualan per Kota:")
print(star.groupby("kota")["total_harga"].sum())

# Total penjualan per segmen pelanggan
print("\nTotal Penjualan per Segmen:")
print(star.groupby("segmen")["total_harga"].sum())


---
## LANGKAH 3: IMPLEMENTASI SKEMA SNOWFLAKE
Pada skema Snowflake, dimensi produk dinormalisasi menjadi dua tabel: `dim_produk` (memegang FK ke kategori) dan `dim_kategori` (menyimpan rincian kategori dan divisi).


In [ ]:
# Tabel kategori (dimensi tingkat 2 hasil normalisasi)
dim_kategori = pd.DataFrame({
    "id_kategori": [10, 20],
    "kategori":    ["Elektronik", "Perlengkapan"],
    "divisi":      ["Teknologi", "Logistik"],
})

# dim_produk memegang FK id_kategori
dim_produk_snow = pd.DataFrame({
    "id_produk":    [1, 2, 3],
    "nama_produk":  ["Laptop", "Printer", "Tinta"],
    "id_kategori":  [10, 10, 20],
    "harga_satuan": [8000000, 1500000, 25000],
})

# Join berantai: fakta -> dim_produk_snow -> dim_kategori
snow = sales.merge(dim_produk_snow, on="id_produk", suffixes=("", "_dim"))
snow = snow.merge(dim_kategori, on="id_kategori")
snow["total_harga"] = snow["jumlah"] * snow["harga_satuan"]

snow[["id_transaksi", "nama_produk", "kategori", "divisi", "jumlah", "total_harga"]]


### Agregasi Roll-Up pada Skema Snowflake


In [ ]:
# Roll-up: Total penjualan per divisi
print("Total Penjualan per Divisi:")
print(snow.groupby("divisi")["total_harga"].sum())


---
## LANGKAH 4: OPERASI OLAP (ROLL-UP, DRILL-DOWN, SLICE, DICE)
Eksplorasi data warehouse dengan kubus data multidimensi lengkap.


In [ ]:
# Kubus data lengkap (menggabungkan fakta dengan seluruh dimensi)
kubus = sales.merge(dim_pelanggan, on="id_pelanggan")
kubus = kubus.merge(dim_produk,  on="id_produk", suffixes=("", "_dim"))
kubus = kubus.merge(dim_waktu,   on="id_waktu")
kubus["total_harga"] = kubus["jumlah"] * kubus["harga_satuan"]

kubus[["id_transaksi", "nama", "kota", "nama_produk", "kategori", "total_harga"]]


In [ ]:
# 4.1 Roll-up (meringkas ke level yang lebih tinggi: kota)
print("ROLL-UP — penjualan per kota:")
print(kubus.groupby("kota")["total_harga"].sum())

# 4.2 Drill-down (merinci ke level yang lebih rendah: segmen)
print("\nDRILL-DOWN — penjualan per segmen pelanggan:")
print(kubus.groupby("segmen")["total_harga"].sum())

# Kubus 2D (Pivot Table: Kota x Kategori Produk)
print("\nKUBUS 2D — total penjualan per kota x kategori:")
display(kubus.pivot_table(index="kota", columns="kategori", values="total_harga", aggfunc="sum", fill_value=0))

# 4.3 Slice (mengambil irisan khusus kota Semarang)
print("\nSLICE — hanya Semarang:")
iris = kubus[kubus["kota"] == "Semarang"]
display(iris[["id_transaksi", "nama", "total_harga"]])

# 4.4 Dice (sub-kubus: kota Semarang DAN total_harga > 500.000)
print("\nDICE — Semarang & total > 500 rb:")
hasil_dice = kubus[(kubus["kota"] == "Semarang") & (kubus["total_harga"] > 500000)]
display(hasil_dice[["id_transaksi", "nama", "total_harga"]])


---
## LANGKAH 5: MENYIMPAN DATA WAREHOUSE KE SQLITE & QUERY JOIN
Menyimpan tabel ke DBMS relasional SQLite untuk kebutuhan persistensi data ETL.


In [ ]:
import sqlite3

con = sqlite3.connect("data_warehouse.db")
sales.to_sql("fact_sales", con, if_exists="replace", index=False)
dim_pelanggan.to_sql("dim_pelanggan", con, if_exists="replace", index=False)
dim_produk.to_sql("dim_produk", con, if_exists="replace", index=False)
dim_waktu.to_sql("dim_waktu", con, if_exists="replace", index=False)

print("Tabel berhasil disimpan ke data_warehouse.db!")

# Query SQL JOIN murni
hasil_sql = pd.read_sql_query("""
    SELECT s.id_transaksi, p.nama_produk, p.kategori, s.jumlah, s.total_harga
    FROM fact_sales AS s
    JOIN dim_produk AS p ON s.id_produk = p.id_produk
""", con)

display(hasil_sql)
con.close()


---
# STUDI KASUS / TUGAS PRAKTIKUM: TOKO ONLINE "ELEKTROSHOP"
**Lingkup Waktu:** Januari – Maret 2026 (Kuartal 1 / Q1 2026)  
Toko online **ElektroShop** menjual beragam produk elektronik dan aksesoris komputer.


In [ ]:
# 1. Membangun Tabel Dimensi ElektroShop
dim_pelanggan_es = pd.DataFrame({
    "idpelanggan": [1, 2, 3, 4, 5],
    "nama":        ["Ahmad Dahlan", "Bela Pertiwi", "Chandra Wijaya", "Diana Lestari", "Eko Prasetyo"],
    "kota":        ["Semarang", "Solo", "Semarang", "Yogyakarta", "Surabaya"],
    "provinsi":    ["Jawa Tengah", "Jawa Tengah", "Jawa Tengah", "D.I. Yogyakarta", "Jawa Timur"],
    "segmen":      ["Retail", "Corporate", "Retail", "Wholesale", "Corporate"]
})

dim_produk_es = pd.DataFrame({
    "idproduk":    [101, 102, 103, 104, 105],
    "nama_produk": ["Laptop ASUS ROG", "Smart TV Samsung 43"", "Mouse Wireless Logitech", "Keyboard Mechanical RGB", "Smartphone Xiaomi 13"],
    "kategori":    ["Komputer & Laptop", "Elektronik Rumah", "Aksesoris PC", "Aksesoris PC", "Telekomunikasi"],
    "harga":       [14000000, 4500000, 150000, 450000, 3200000]
})

dim_waktu_es = pd.DataFrame({
    "idwaktu": [1, 2, 3, 4, 5, 6],
    "tanggal": ["2026-01-10", "2026-01-25", "2026-02-05", "2026-02-18", "2026-03-08", "2026-03-24"],
    "tahun":   [2026, 2026, 2026, 2026, 2026, 2026],
    "bulan":   ["Januari", "Januari", "Februari", "Februari", "Maret", "Maret"],
    "kuartal": ["Q1", "Q1", "Q1", "Q1", "Q1", "Q1"]
})

# 2. Membangun Tabel Fakta Penjualan (14 Transaksi)
fakta_penjualan_es = pd.DataFrame({
    "idtransaksi": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014],
    "idpelanggan": [1,    2,    3,    4,    5,    1,    2,    3,    4,    5,    1,    3,    4,    2],
    "idproduk":    [101,  103,  102,  104,  105,  104,  101,  105,  103,  102,  103,  101,  105,  104],
    "idwaktu":     [1,    1,    2,    2,    3,    3,    4,    4,    5,    5,    6,    6,    6,    6],
    "jumlah":      [1,    10,   1,    8,    2,    2,    2,    1,    5,    3,    4,    1,    3,    5],
})

# Hitung Measure total_harga
harga_dict = dict(zip(dim_produk_es["idproduk"], dim_produk_es["harga"]))
fakta_penjualan_es["total_harga"] = fakta_penjualan_es["jumlah"] * fakta_penjualan_es["idproduk"].map(harga_dict)
display(fakta_penjualan_es)


In [ ]:
# 3. Skema Star ElektroShop & Agregasi
star_es = fakta_penjualan_es.merge(dim_pelanggan_es, on="idpelanggan") \
                            .merge(dim_produk_es, on="idproduk") \
                            .merge(dim_waktu_es, on="idwaktu")

print("--- Total Penjualan per Kota ---")
display(star_es.groupby("kota")["total_harga"].sum().reset_index())

print("--- Total Penjualan per Kategori Produk ---")
display(star_es.groupby("kategori")["total_harga"].sum().reset_index())


In [ ]:
# 4. Skema Snowflake ElektroShop (Normalisasi Divisi)
dim_kategori_es = pd.DataFrame({
    "idkategori": [201, 202, 203, 204],
    "kategori":   ["Komputer & Laptop", "Elektronik Rumah", "Aksesoris PC", "Telekomunikasi"],
    "divisi":     ["Divisi IT Hardware", "Divisi Consumer Electronics", "Divisi IT Hardware", "Divisi Gadget & Mobile"]
})

dim_produk_snow_es = pd.DataFrame({
    "idproduk":    [101, 102, 103, 104, 105],
    "nama_produk": ["Laptop ASUS ROG", "Smart TV Samsung 43"", "Mouse Wireless Logitech", "Keyboard Mechanical RGB", "Smartphone Xiaomi 13"],
    "idkategori":  [201, 202, 203, 203, 204],
    "harga":       [14000000, 4500000, 150000, 450000, 3200000]
})

snow_es = fakta_penjualan_es.merge(dim_produk_snow_es, on="idproduk") \
                            .merge(dim_kategori_es, on="idkategori")

print("--- Roll-Up: Total Penjualan per Divisi ---")
display(snow_es.groupby("divisi")["total_harga"].sum().reset_index())


In [ ]:
# 5. Operasi OLAP & Interpretasi
print("[5.A] DRILL-DOWN: Total Penjualan per Segmen Pelanggan")
display(star_es.groupby("segmen")["total_harga"].sum().reset_index())

print("\n[5.B] SLICE: Transaksi di Kota Semarang")
slice_es = star_es[star_es["kota"] == "Semarang"]
display(slice_es[["idtransaksi", "nama", "kota", "nama_produk", "jumlah", "total_harga"]])

print("\n[5.C] DICE: Transaksi dengan Total >= Rp 1.000.000")
dice_es = star_es[star_es["total_harga"] >= 1000000]
display(dice_es[["idtransaksi", "nama", "kota", "nama_produk", "segmen", "total_harga"]])


In [ ]:
# 6. Simpan Seluruh Tabel ke Database SQLite elektroshop.db & Query JOIN
con_es = sqlite3.connect("elektroshop.db")
fakta_penjualan_es.to_sql("fakta_penjualan", con_es, if_exists="replace", index=False)
dim_pelanggan_es.to_sql("dim_pelanggan", con_es, if_exists="replace", index=False)
dim_produk_es.to_sql("dim_produk", con_es, if_exists="replace", index=False)
dim_waktu_es.to_sql("dim_waktu", con_es, if_exists="replace", index=False)
dim_kategori_es.to_sql("dim_kategori", con_es, if_exists="replace", index=False)

print("Tabel berhasil disimpan ke elektroshop.db!")

# Query SQL JOIN: Nama Produk dan Total Penjualannya
sql_join_es = """
    SELECT 
        p.nama_produk,
        p.kategori,
        SUM(f.jumlah) AS total_unit_terjual,
        SUM(f.total_harga) AS total_nominal_penjualan
    FROM fakta_penjualan AS f
    JOIN dim_produk AS p ON f.idproduk = p.idproduk
    GROUP BY p.idproduk, p.nama_produk, p.kategori
    ORDER BY total_nominal_penjualan DESC
"""

hasil_sql_es = pd.read_sql_query(sql_join_es, con_es)
display(hasil_sql_es)
con_es.close()


---
### Rangkuman & Kesimpulan
1. **Data Warehouse** memisahkan pengukuran numerik (*measures*) pada **Tabel Fakta** dan konteks analitis pada **Tabel Dimensi**.
2. **Skema Star** memprioritaskan performa query analitis yang cepat dan relasi sederhana satu tingkat (*denormalized*).
3. **Skema Snowflake** melakukan normalisasi dimensi bertingkat untuk menghemat ruang penyimpanan data dan mencegah redundansi.
4. **OLAP** memungkinkan eksekutif dan analis bisnis mengeksplorasi data penjualan melalui operasi *Roll-up, Drill-down, Slice,* dan *Dice*.
5. Persistensi ke database relasional seperti **SQLite** menjadi fondasi penyimpanan pipeline ETL (*Extract, Transform, Load*).
